# Compliance Annex — Access Code Investigation

The Arturic Industries **Compliance Annex** is locked behind an access code.  
The only hint provided: *"consult your facility photograph for orientation."*

This notebook documents the investigation step by step:
1. Extract GPS coordinates from the facility photograph
2. Visit the location — photograph the signage on site
3. Build candidates directly from what the signs say
4. Verify against the SHA-256 gate

## 1. Gate mechanics

Inspecting `compliance.html` reveals a simple SHA-256 check:

```js
const H      = "38b19f2e76c9fa1e3ab74c80fb3e95b3cd761ce39b0e2359b6ac15e012220907";
const SUFFIX = "arturic";

function verifyAccess(raw) {
    const normalized = raw.trim().toUpperCase();
    const digest     = sha256(normalized);
    const match      = constantTimeEqual(digest, H);
    const nextPage   = match ? sha256(normalized + SUFFIX) + ".html" : null;
    return { match, nextPage };
}
```

In [ ]:
import hashlib, hmac

H      = "38b19f2e76c9fa1e3ab74c80fb3e95b3cd761ce39b0e2359b6ac15e012220907"
SUFFIX = "arturic"

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

def verify(raw: str) -> dict:
    normalized = raw.strip().upper()
    digest     = sha256_hex(normalized)
    match      = hmac.compare_digest(digest, H)
    next_page  = sha256_hex(normalized + SUFFIX) + ".html" if match else None
    return {"candidate": raw, "normalized": normalized, "match": match, "next_page": next_page}

print("Gate helpers ready.")

## 2. Extracting GPS from the facility photograph

The welcome packet states: *"all layers are authentic and unaltered"* — a hint that the image carries embedded metadata.

In [ ]:
from pathlib import Path
from PIL import Image, ExifTags
from IPython.display import display

FACILITY_IMG = Path("../facility_exterior.png")
img = Image.open(FACILITY_IMG)
display(img.resize((800, int(800 * img.height / img.width))))

In [ ]:
def dms_to_decimal(dms, ref: str) -> float:
    d, m, s = (float(x) for x in dms)
    decimal  = d + m / 60 + s / 3600
    return -decimal if ref in ("S", "W") else decimal

exif = img.getexif()
gps  = exif.get_ifd(ExifTags.IFD.GPSInfo)

print(f"Format : {img.format} | EXIF tags: {len(exif)} | GPS tags: {len(gps)}")
for tag_id, value in gps.items():
    print(f"  {ExifTags.GPSTAGS.get(tag_id, tag_id)}: {value}")

lat = dms_to_decimal(gps[2], gps[1])
lon = dms_to_decimal(gps[4], gps[3])
print(f"\nDecimal coords : {lat:.6f}, {lon:.6f}")
print(f"Google Maps    : https://www.google.com/maps?q={lat:.6f},{lon:.6f}")

## 3. On-site investigation

Coordinates **40.3652°N, 74.1639°W** → **Holmdel, New Jersey** — the **Bell Works** campus (formerly Bell Telephone Laboratories, designed by Eero Saarinen, 1962).

A visit to the site reveals two signs that point directly to the answer.

---

### Sign 1 — Radio Astronomy marker (on the grounds)

![Radio Astronomy sign](sign_radio_astronomy.png)

> **RADIO ASTRONOMY**  
> On this site in 1932, Bell Labs scientist **Karl Jansky** first discovered radio waves coming from outer space, thus beginning the science of radio astronomy.

---

### Sign 2 — Exits (Bell Works entrance)

![Exits sign](sign_exits.png)

> **EXITS**  
> ← CRAWFORDS CRN RD  
> ↑ MIDDLETOWN RD  
> ↑ ROBERTS RD  
> *Bell Works*

---

### Severance connection

Bell Works is not just historically significant — it is also the **exterior filming location for Lumon Industries** in the Apple TV+ series *Severance*. The show depicts a corporation that forces employees to surgically split their work and personal memories — and uses this exact building as its headquarters.

The parallel with Arturic Industries is deliberate: the "Macro Data Refinement" division, the bins (Grief, Bliss, Anxiety, Spite), the Founder's Creed, the cryptic compliance rules — all mirror Severance's Lumon Industries almost exactly. The challenge is a puzzle dressed as a job application.

## 4. Candidate list — derived directly from the signs

Every candidate below comes from text visible on-site or from its direct historical context.  
No guessing, no brute force.

In [ ]:
candidates = [
    # ── From Sign 1: Radio Astronomy marker ──────────────────────────────────
    "RADIO ASTRONOMY",       # the title of the sign
    "RADIOASTRONOMY",
    "KARL JANSKY",           # name on the sign
    "JANSKY",                # surname only
    "KARLJANSKY",
    "1932",                  # year on the sign
    "RADIO WAVES",           # what Jansky discovered

    # ── From Sign 2: Exits / Bell Works ──────────────────────────────────────
    "BELL WORKS",            # the building name on the sign
    "CRAWFORDS CRN RD",      # street name on sign
    "CRAWFORDS",
    "MIDDLETOWN RD",
    "MIDDLETOWN",
    "ROBERTS RD",

    # ── From Severance (Bell Works = Lumon Industries exterior) ──────────────
    "LUMON",                 # the Arturic analogue in Severance
    "LUMON INDUSTRIES",
    "KIER",                  # Kier Eagan — Lumon founder (mirrors Aldric Arturic)
    "MACRODATA",             # the MDR division in Severance
    "MACRO DATA",
    "SEVERANCE",
]

## 5. Verify candidates

In [ ]:
import pandas as pd

rows = [verify(c) for c in candidates]
df   = pd.DataFrame(rows)[["candidate", "normalized", "match", "next_page"]]

def highlight_match(row):
    style = "background-color: #c8f7c5; font-weight: bold" if row["match"] else ""
    return [style] * len(row)

df.style.apply(highlight_match, axis=1)

## 6. Result

In [ ]:
winner = next((r for r in rows if r["match"]), None)

if winner:
    print(f"✓ Access code : {winner['candidate']}")
    print(f"  Normalized  : {winner['normalized']}")
    print(f"  Next page   : {winner['next_page']}")
else:
    print("✗ No match — expand the candidate list.")